In [13]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import optuna

# -------------------------
# 1. Load data (ONLY ONCE)
# -------------------------
df = pd.read_csv('stressdata_preprocessed.csv')

y = df['Stress Level'].values - 1
X = df.drop('Stress Level', axis=1).values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------
# 2. Model builder
# -------------------------
def build_model(trial):
    model = keras.Sequential()

    # Input layer
    model.add(layers.Input(shape=(X.shape[1],)))

    # Hidden layers
    n_layers = trial.suggest_int("hidden_layers", 1, 3)

    for i in range(n_layers):
        units = trial.suggest_int(f"units_l{i}", 16, 128)
        dropout = trial.suggest_float(f"dropout_l{i}", 0.0, 0.5)

        model.add(layers.Dense(units, activation="relu"))
        model.add(layers.Dropout(dropout))

    # Output layer
    model.add(layers.Dense(10))  # 10 classes

    # Learning rate
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )

    return model

# -------------------------
# 3. Objective function
# -------------------------
def objective(trial):
    model = build_model(trial)

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    _, acc = model.evaluate(X_val, y_val, verbose=0)
    return acc

# -------------------------
# 4. Run optimization
# -------------------------
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best Accuracy:", study.best_value)
print("Best Params:", study.best_params)

# -------------------------
# 5. Final model training
# -------------------------
best = study.best_params

model = keras.Sequential()
model.add(layers.Input(shape=(X.shape[1],)))

for i in range(best["hidden_layers"]):
    model.add(layers.Dense(best[f"units_l{i}"], activation="relu"))
    model.add(layers.Dropout(best[f"dropout_l{i}"]))

model.add(layers.Dense(10))

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=best["lr"]),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)

[I 2026-04-26 16:01:38,694] A new study created in memory with name: no-name-61fdbaa8-e4e5-4876-a837-d4aaf4857d3f
[I 2026-04-26 16:01:46,988] Trial 0 finished with value: 0.1733333319425583 and parameters: {'hidden_layers': 2, 'units_l0': 53, 'dropout_l0': 0.4855321984529716, 'units_l1': 43, 'dropout_l1': 0.4156199141333766, 'lr': 3.363629578918785e-05}. Best is trial 0 with value: 0.1733333319425583.
[I 2026-04-26 16:01:55,570] Trial 1 finished with value: 0.2199999988079071 and parameters: {'hidden_layers': 2, 'units_l0': 102, 'dropout_l0': 0.13406301413715288, 'units_l1': 69, 'dropout_l1': 0.06988792839972696, 'lr': 0.0003742036183687098}. Best is trial 1 with value: 0.2199999988079071.
[I 2026-04-26 16:02:03,752] Trial 2 finished with value: 0.20000000298023224 and parameters: {'hidden_layers': 1, 'units_l0': 68, 'dropout_l0': 0.39618543945230106, 'lr': 0.0003920669942618647}. Best is trial 1 with value: 0.2199999988079071.
[I 2026-04-26 16:02:11,317] Trial 3 finished with value: 0

Best Accuracy: 0.25999999046325684
Best Params: {'hidden_layers': 3, 'units_l0': 17, 'dropout_l0': 0.0359519081894169, 'units_l1': 91, 'dropout_l1': 0.011417252320557963, 'units_l2': 126, 'dropout_l2': 0.48774386943402964, 'lr': 0.009745908609346788}
Epoch 1/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.1600 - loss: 2.2221
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2025 - loss: 2.0196
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2317 - loss: 1.9648
Epoch 4/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2350 - loss: 1.9062
Epoch 5/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2417 - loss: 1.8858
Epoch 6/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2508 - loss: 1.8602
Epoch 7/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2650 - loss: 1.8336
Epoch 8/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2817 - loss: 1.8380
Epoch 9/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2742 - loss: 

In [14]:
model.save("stress_model.keras")
